<a href="https://colab.research.google.com/github/krishgupta1843-star/project-beach/blob/main/Customer_Churn_DeepLearning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.metrics import (confusion_matrix, classification_report,
                              accuracy_score)
tf.random.set_seed(42)
np.random.seed(42)
print(f'   TensorFlow version : {tf.__version__}')
print(f'   Keras version      : {keras.__version__}')

In [ ]:
df = pd.read_csv('Churn_Modelling.csv')
print(f'✅ Dataset loaded!')
print(f'   Rows    : {df.shape[0]}')
print(f'   Columns : {df.shape[1]}')
print(f'\n── Column Names ──')
print(df.columns.tolist())
print('\n── Data Types ──')
print(df.dtypes)
print('\n── First 5 Rows ──')
display(df.head())
print('── Last 5 Rows ──')
display(df.tail())

In [ ]:
print('═' * 60)

print('═' * 60)
print('\n── 1. Dataset Info ──')
df.info(memory_usage='deep')

display(df.describe())
missing = pd.DataFrame({
    'Count'  : df.isnull().sum(),
    'Percent': (df.isnull().sum() / len(df) * 100).round(2)
}).sort_values('Percent', ascending=False)
print(missing)
print(f'\n── 4. Duplicate Rows: {df.duplicated().sum()} ──')

print(df.nunique())
print(df['Exited'].value_counts())
print(f'   Stayed  (0): {(df["Exited"]==0).sum()} ({(df["Exited"]==0).mean()*100:.1f}%)')
print(f'   Churned (1): {(df["Exited"]==1).sum()} ({(df["Exited"]==1).mean()*100:.1f}%)')
mem = df.memory_usage(deep=True).sum()
print(f'\n── 7. Total Memory Usage: {mem/1024:.1f} KB ──')

In [ ]:

df_clean = df.copy()

print('═' * 60)
print('  RULE 1 — Drop columns with > 20% missing values')
print('═' * 60)
missing_pct  = df_clean.isnull().sum() / len(df_clean) * 100
cols_to_drop = missing_pct[missing_pct > 20].index.tolist()
if cols_to_drop:
    for c in cols_to_drop:
        print(f'   Dropped: {c}  ({missing_pct[c]:.1f}% missing)')
    df_clean.drop(columns=cols_to_drop, inplace=True)
else:
    print('No column exceeds 20% missing. All columns retained.')

In [ ]:
print('═' * 60)
print('  RULE 2 — Fill remaining missing values')
print('═' * 60)
num_cols = df_clean.select_dtypes(include='number').columns
for col in num_cols:
    if df_clean[col].isnull().any():
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())
        print(f' {col}: filled with median')
cat_cols = df_clean.select_dtypes(include='object').columns
for col in cat_cols:
    if df_clean[col].isnull().any():
        df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])
        print(f' {col}: filled with mode')

if df_clean.isnull().sum().sum() == 0:
    print('  No missing values existed — nothing to fill')

print('\n═' * 31)
print('  RULE 3 — Remove duplicate rows')
print('═' * 60)
before = len(df_clean)
df_clean = df_clean.drop_duplicates()
print(f'  Duplicates removed: {before - len(df_clean)}')

print('\n═' * 31)
print('  RULE 4 — Standardise column names')
print('═' * 60)
df_clean.columns = (
    df_clean.columns
    .str.strip().str.lower()
    .str.replace(' ', '_', regex=False)
    .str.replace('[^a-z0-9_]', '', regex=True)
)
print(f'Column names: {df_clean.columns.tolist()}')

print('\n═' * 31)
print('  RULE 5 — Strip spaces from categorical values')
print('═' * 60)
for col in df_clean.select_dtypes(include='object').columns:
    df_clean[col] = df_clean[col].astype(str).str.strip()
print('\n═' * 31)
print('  RULE 6 — Final null verification')
print('═' * 60)
print(f'  Total nulls remaining : {df_clean.isnull().sum().sum()}')
print(f'  Final shape           : {df_clean.shape}')


In [ ]:
print('═' * 60)
print('  REDUNDANT COLUMN CHECK')
print('═' * 60)
id_cols = ['rownumber', 'customerid', 'surname']
existing_id_cols = [c for c in id_cols if c in df_clean.columns]


for c in existing_id_cols:
    reason = {
        'rownumber'  : 'Serial number — does not describe the customer',
        'customerid' : 'Unique ID — has no relationship with churn behaviour',
        'surname'    : "Customer's name — personal identifier, no predictive value"
    }
    print(f'  Dropped: {c:15s} → {reason[c]}')

df_clean.drop(columns=existing_id_cols, inplace=True)
print('\n── High Correlation Check (|r| > 0.95) ──')
num_df   = df_clean.select_dtypes(include='number').drop(columns=['exited'], errors='ignore')
corr_mat = num_df.corr().abs()
upper    = corr_mat.where(np.triu(np.ones(corr_mat.shape), k=1).astype(bool))
redundant = [c for c in upper.columns if any(upper[c] > 0.95)]
if redundant:
    df_clean.drop(columns=redundant, inplace=True)
    print(f'  ✂  Dropped: {redundant}')
else:
    print(' No highly correlated pairs found.')

print(f'\n Final shape: {df_clean.shape}')
print(f'   Remaining columns: {df_clean.columns.tolist()}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Target Variable — Churn Distribution', fontsize=14, fontweight='bold')

sns.countplot(data=df_clean, x='exited', palette=['steelblue','tomato'], ax=axes[0])
axes[0].set_xticklabels(['Stayed (0)', 'Churned (1)'])
axes[0].set_title('Count')
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height())}',
        (p.get_x() + p.get_width()/2, p.get_height()),
        ha='center', va='bottom')

df_clean['exited'].value_counts().plot.pie(
    ax=axes[1], autopct='%1.1f%%',
    labels=['Stayed', 'Churned'],
    colors=['steelblue', 'tomato'], startangle=90)
axes[1].set_ylabel('')
axes[1].set_title('Percentage')

plt.tight_layout()
plt.show()
print('💡 Insight: 79.6% customers stayed vs 20.4% churned. Dataset is imbalanced — typical for banking.')

In [ ]:
num_features = ['creditscore', 'age', 'tenure', 'balance', 'numofproducts', 'estimatedsalary']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Distribution of Numerical Features', fontsize=14, fontweight='bold')

for ax, col in zip(axes.flatten(), num_features):
    sns.histplot(df_clean[col], kde=True, ax=ax, color='steelblue', bins=25)
    ax.set_title(col.replace('_',' ').title(), fontsize=10)
    ax.set_xlabel('')

plt.tight_layout()
plt.show()
print(' Insight: Age has a right skew. Balance has a spike at 0 (many customers with zero balance). Salary is uniformly distributed.')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Boxplots by Churn Status (0=Stayed, 1=Churned)', fontsize=14, fontweight='bold')

for ax, col in zip(axes.flatten(), num_features):
    sns.boxplot(data=df_clean, x='exited', y=col,
                palette=['steelblue','tomato'], ax=ax)
    ax.set_title(col.replace('_',' ').title(), fontsize=10)
    ax.set_xticklabels(['Stayed','Churned'])

plt.tight_layout()
plt.show()
print(' Insight: Churned customers are significantly older. They also tend to have higher balances — possibly inactive older customers.')

In [ ]:
cat_features = ['geography', 'gender', 'hascrcard', 'isactivemember']

fig, axes = plt.subplots(1, 4, figsize=(16, 5))
fig.suptitle('Categorical Features vs Churn', fontsize=14, fontweight='bold')

for ax, col in zip(axes, cat_features):
    sns.countplot(data=df_clean, x=col, hue='exited',
                  palette=['steelblue','tomato'], ax=ax)
    ax.set_title(col.replace('_',' ').title(), fontsize=10)
    ax.legend(title='Exited', labels=['Stayed','Churned'])
    ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()
print('💡 Insight: Germany has the highest churn rate. Inactive members churn much more than active ones. Females churn slightly more than males.')

In [ ]:
plt.figure(figsize=(10, 7))
corr = df_clean.select_dtypes(include='number').corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', mask=mask,
            cmap='coolwarm', linewidths=0.5, annot_kws={'size': 9})
plt.title('Correlation Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print('💡 Insight: Age has the highest positive correlation with Exited (0.29). NumOfProducts has a slight negative correlation.')

In [ ]:
print('Generating Pairplot... (may take a few seconds)')
pair_cols = ['creditscore', 'age', 'balance', 'estimatedsalary', 'exited']
pair_df   = df_clean[pair_cols].copy()
pair_df['exited'] = pair_df['exited'].map({0: 'Stayed', 1: 'Churned'})

sns.pairplot(pair_df, hue='exited',
             palette={'Stayed': 'steelblue', 'Churned': 'tomato'},
             plot_kws={'alpha': 0.3, 's': 10}, diag_kind='kde')
plt.suptitle('Pairplot — Churned vs Stayed', y=1.01, fontsize=13, fontweight='bold')
plt.show()
print('💡 Insight: Age is the clearest separator — churned customers (red) cluster at higher ages.')

In [ ]:
print('═' * 60)
print('  FEATURE ENGINEERING')
print('═' * 60)
df_clean['zero_balance'] = (df_clean['balance'] == 0).astype(int)
print('   Created: zero_balance  (1 if Balance = 0, else 0)')
print(f'     Customers with zero balance: {df_clean["zero_balance"].sum()}')
df_clean['balance_per_product'] = df_clean['balance'] / (df_clean['numofproducts'] + 1)
print('   Created: balance_per_product  (Balance divided by number of products)')
df_clean['age_group'] = pd.cut(df_clean['age'],
                                bins=[0, 30, 45, 60, 100],
                                labels=['Young', 'Middle', 'Senior', 'Elder'])
print('   Created: age_group  (Young <30, Middle 30-45, Senior 45-60, Elder >60)')

df_clean['credit_category'] = pd.cut(df_clean['creditscore'],
                                      bins=[0, 580, 670, 740, 800, 900],
                                      labels=['Poor', 'Fair', 'Good', 'Very Good', 'Exceptional'])
print(' Created: credit_category  (Standard credit score bands)')

print(f'\n  Shape after feature engineering: {df_clean.shape}')

In [ ]:
TARGET = 'exited'

X = df_clean.drop(columns=[TARGET])
y = df_clean[TARGET]

print('✅ Features and Target separated!')
print(f'   X shape : {X.shape}')
print(f'   y shape : {y.shape}')
print(f'\n   Feature columns: {X.columns.tolist()}')
print(f'\n   Target distribution:')
print(f'   Stayed (0)  : {(y==0).sum()}')
print(f'   Churned (1) : {(y==1).sum()}')

In [ ]:
X_encoded = X.copy()
le = LabelEncoder()
X_encoded['gender'] = le.fit_transform(X_encoded['gender'].astype(str))
print(f' Label Encoded: gender  → {dict(zip(le.classes_, le.transform(le.classes_)))}')

age_map    = {'Young': 0, 'Middle': 1, 'Senior': 2, 'Elder': 3}
credit_map = {'Poor': 0, 'Fair': 1, 'Good': 2, 'Very Good': 3, 'Exceptional': 4}

X_encoded['age_group']      = X_encoded['age_group'].map(age_map)
X_encoded['credit_category'] = X_encoded['credit_category'].map(credit_map)
print('   Label Encoded: age_group and credit_category (ordered categories)')
X_encoded = pd.get_dummies(X_encoded, columns=['geography'], drop_first=True)

bool_cols = X_encoded.select_dtypes(include='bool').columns
X_encoded[bool_cols] = X_encoded[bool_cols].astype(int)

print(f' One-Hot Encoded: geography → {[c for c in X_encoded.columns if "geography" in c]}')

print(f'\n  Shape before encoding: {X.shape}')
print(f'  Shape after encoding : {X_encoded.shape}')
print(f'  All columns: {X_encoded.columns.tolist()}')

In [ ]:
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_encoded)
X_scaled = pd.DataFrame(X_scaled, columns=X_encoded.columns)

print('✅ StandardScaler applied!')
print(f'   Features scaled : {X_scaled.shape[1]}')
print(f'   All features now have mean ≈ 0 and std ≈ 1')
print('\n   Sample (first 3 rows, first 6 cols):')
display(X_scaled.iloc[:3, :6].round(4))

In [ ]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X_scaled, y, test_size=0.15, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.15/0.85,
    random_state=42, stratify=y_temp)
print(f'   Training   : {X_train.shape[0]} samples  (70%)')
print(f'   Validation : {X_val.shape[0]} samples  (15%)')
print(f'   Test       : {X_test.shape[0]} samples  (15%)')
print(f'\n   Churn rate in Training   : {y_train.mean()*100:.1f}%')
print(f'   Churn rate in Validation : {y_val.mean()*100:.1f}%')
print(f'   Churn rate in Test       : {y_test.mean()*100:.1f}%')

In [ ]:
input_dim = X_train.shape[1]
print(f'Input features: {input_dim}')
model = Sequential(name='Customer_Churn_ANN')

model.add(Dense(128, activation='relu', input_shape=(input_dim,),
                name='Hidden_Layer_1'))
model.add(BatchNormalization(name='BatchNorm_1'))
model.add(Dropout(0.3, name='Dropout_1'))

model.add(Dense(64, activation='relu', name='Hidden_Layer_2'))
model.add(BatchNormalization(name='BatchNorm_2'))
model.add(Dropout(0.3, name='Dropout_2'))

model.add(Dense(32, activation='relu', name='Hidden_Layer_3'))
model.add(Dropout(0.2, name='Dropout_3'))

model.add(Dense(1, activation='sigmoid', name='Output_Layer'))

model.summary()

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print('✅ Model compiled!')
print('   Optimizer : Adam  (learning_rate = 0.001)')
print('   Loss      : binary_crossentropy')
print('   Metrics   : accuracy')

In [ ]:
# ── Callbacks ─────────────────────────────────────────────────────────────────
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True,
    verbose=1
)

checkpoint = ModelCheckpoint(
    'best_churn_model.keras',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop, checkpoint],
    verbose=1
)


In [ ]:

test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f'✅ Test Evaluation:')
print(f'   Test Accuracy : {test_acc*100:.2f}%')
print(f'   Test Loss     : {test_loss:.4f}')


y_prob  = model.predict(X_test).flatten()
y_pred  = (y_prob >= 0.5).astype(int)

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Stayed', 'Churned'],
            yticklabels=['Stayed', 'Churned'])
plt.title('Confusion Matrix — ANN on Test Data', fontweight='bold')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred,
      target_names=['Stayed (0)', 'Churned (1)']))
print(f'Overall Accuracy : {accuracy_score(y_test, y_pred)*100:.2f}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('ANN Training History', fontsize=14, fontweight='bold')

axes[0].plot(history.history['accuracy'],     label='Train Accuracy', color='steelblue')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy',   color='tomato')
axes[0].set_title('Accuracy over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(history.history['loss'],     label='Train Loss', color='steelblue')
axes[1].plot(history.history['val_loss'], label='Val Loss',   color='tomato')
axes[1].set_title('Loss over Epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.show()

final_train_acc = history.history['accuracy'][-1]
final_val_acc   = history.history['val_accuracy'][-1]
gap = abs(final_train_acc - final_val_acc)

print(f'Final Training Accuracy   : {final_train_acc*100:.2f}%')
print(f'Final Validation Accuracy : {final_val_acc*100:.2f}%')
print(f'Gap                       : {gap*100:.2f}%')
print()
if gap < 0.03:
    print('✅ GOOD FIT — Training and validation accuracy are close. Model generalises well.')
elif final_train_acc > final_val_acc + 0.05:
    print('⚠️  OVERFITTING — Model performs much better on training than validation.')
else:
    print('⚠️  UNDERFITTING — Model needs more capacity or training time.')

In [ ]:
y_probabilities = model.predict(X_test).flatten()
y_predicted     = (y_probabilities >= 0.5).astype(int)  # Threshold = 0.5

pred_df = pd.DataFrame({
    'Actual'           : y_test.values[:10],
    'Predicted'        : y_predicted[:10],
    'Probability (%)' : (y_probabilities[:10] * 100).round(2)
})
pred_df['Actual Label']    = pred_df['Actual'].map({0: 'Stayed', 1: 'Churned'})
pred_df['Predicted Label'] = pred_df['Predicted'].map({0: 'Stayed', 1: 'Churned'})
pred_df['Correct?']        = (pred_df['Actual'] == pred_df['Predicted']).map({True: '✅', False: '❌'})

print(f'── First 10 Predictions ──')
display(pred_df[['Actual Label','Predicted Label','Probability (%)','Correct?']])

correct = (y_predicted == y_test.values).sum()
print(f'\nCorrect predictions on full test set: {correct} / {len(y_test)}')

In [ ]:

model.save('customer_churn_ann.keras')
print('✅ Model saved as: customer_churn_ann.keras')

import joblib
joblib.dump(scaler, 'churn_scaler.pkl')
print('✅ Scaler saved as: churn_scaler.pkl')

print('\n── How to reload and use later ──')
print("""
  from tensorflow import keras
  import joblib

  model  = keras.models.load_model('customer_churn_ann.keras')
  scaler = joblib.load('churn_scaler.pkl')

  # new_customer = scaler.transform(new_data)
  # churn_prob   = model.predict(new_customer)
  # will_churn   = (churn_prob >= 0.5).astype(int)
""")

In [ ]:
final_acc = accuracy_score(y_test, y_pred) * 100

print('═' * 65)
print('  CUSTOMER CHURN PREDICTION — INTERNSHIP REPORT')
print('═' * 65)

print('\n  DATASET SUMMARY')
print('  ├─ Dataset        : Churn_Modelling.csv')
print('  ├─ Total Rows     : 10,000 customers')
print('  ├─ Total Columns  : 14 (after cleaning: 11 features)')
print('  ├─ Problem Type   : Binary Classification')
print('  └─ Target         : Exited (0 = Stayed, 1 = Churned)')

print('\n  DATA CLEANING SUMMARY')
print('  ├─ No missing values — dataset was already clean')
print('  ├─ No duplicate rows')
print('  ├─ Dropped: RowNumber, CustomerId, Surname (identifiers)')
print('  └─ Column names standardised to lowercase')

print('\n  EDA SUMMARY')
print('  ├─ 20.4% customers churned (imbalanced dataset)')
print('  ├─ Age is the #1 predictor of churn')
print('  ├─ Germany has highest churn rate by geography')
print('  └─ Inactive members churn significantly more')

print('\n  MODEL ARCHITECTURE')
print('  ├─ Framework      : TensorFlow / Keras')
print('  ├─ Type           : Artificial Neural Network (ANN)')
print('  ├─ Layers         : Input → Dense(128) → BN → Dropout(0.3)')
print('  │                   → Dense(64) → BN → Dropout(0.3)')
print('  │                   → Dense(32) → Dropout(0.2) → Dense(1, Sigmoid)')
print('  ├─ Optimizer      : Adam (lr=0.001)')
print('  ├─ Loss Function  : Binary Crossentropy')
print('  └─ Callbacks      : EarlyStopping + ModelCheckpoint')

print(f'\n  FINAL RESULTS')
print(f'  ├─ Test Accuracy  : {final_acc:.2f}%')
print(f'  ├─ Test Loss      : {test_loss:.4f}')
print(f'  └─ Data Split     : 70% Train | 15% Val | 15% Test')

print('\n  ADVANTAGES')
print('  ├─ ANN learns complex non-linear patterns')
print('  ├─ Dropout and BatchNorm prevent overfitting')
print('  └─ EarlyStopping saves training time automatically')

print('\n  LIMITATIONS')
print('  ├─ Dataset is imbalanced (only 20.4% churn)')
print('  ├─ ANN is a "black box" — difficult to interpret decisions')
print('  └─ Only 10,000 samples — larger data would improve accuracy')

print('\n  FUTURE IMPROVEMENTS')
print('  ├─ Apply SMOTE to handle class imbalance')
print('  ├─ Use class_weight parameter in model.fit')
print('  ├─ Tune hyperparameters with Keras Tuner')
print('  └─ Deploy as a Flask/FastAPI web application')

print('\n═' * 33)
print('  ✅ PROJECT COMPLETE — INTERNSHIP SUBMISSION READY')
print('═' * 65)